# Rossmann Store Sales Modernized

This notebook upgrades the original project into a reproducible, leakage-safe retail forecasting workflow with time-aware validation, stronger business features, model comparison, explainability, and submission creation.

## 1. Problem Framing

The goal is to forecast Rossmann daily store sales in a way that is both Kaggle-ready and realistic for production use. Key upgrades versus the legacy notebook include:

- Replaced the legacy 70k-row sample with the full training history when fitting the modern pipeline.
- Replaced random train/test splitting with expanding-window validation plus a final 6-week holdout.
- Removed forecast-time-invalid leakage risks from the final model, especially `Customers`.
- Added stronger business features for promotions, competition timing, calendar seasonality, and store metadata.
- Added leakage-safe historical aggregate features that are re-fit inside each time split.
- Compared XGBoost, LightGBM, and CatBoost on the same validation design instead of relying on one model.
- Added error slicing, feature importance, SHAP explainability, artifact saving, and a reproducible script.

In [3]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
from rossmann_modernized import (
    build_config,
    locate_data_paths,
    load_datasets,
    build_schema_summary,
    build_feature_availability,
    prepare_store_metadata,
    prepare_model_frame,
    create_validation_windows,
    run_pipeline,
)

config = build_config(Path.cwd())
paths = locate_data_paths(config.project_dir)
raw_train, raw_test, raw_store = load_datasets(paths)
schema_summary = build_schema_summary(raw_train, raw_test, raw_store)
feature_audit = build_feature_availability(raw_train, raw_test, raw_store)

ModuleNotFoundError: No module named 'catboost'

## 2. Data Loading

In [ ]:
schema_summary.query("dataset == 'train'").head(10), feature_audit[['feature', 'available_in_test', 'allowed_in_final_model', 'reason']]

## 3. Cleaning

In [ ]:
store_features = prepare_store_metadata(raw_store)
train_model = prepare_model_frame(raw_train, store_features)
test_model = prepare_model_frame(raw_test, store_features)
train_model[['Date', 'Store', 'Sales', 'Open', 'Promo', 'CompetitionActive', 'Promo2Active']].head()

## 4. Feature Engineering

In [ ]:
selected_columns = [
    'Store', 'StoreType', 'Assortment', 'Promo', 'Promo2', 'StateHoliday', 'SchoolHoliday',
    'CompetitionDistance', 'CompetitionDistanceLog', 'CompetitionActive', 'MonthsSinceCompetitionOpen',
    'Promo2Started', 'Promo2Active', 'WeeksSincePromo2Start', 'PromoIntervalActive',
    'year', 'month', 'day', 'iso_week', 'day_of_week', 'quarter', 'is_weekend',
    'is_month_start', 'is_month_end', 'month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos'
]
train_model[selected_columns].head()

## 5. Validation Design

In [ ]:
holdout_start, pre_holdout, holdout, cv_windows = create_validation_windows(
    train_model, holdout_days=config.holdout_days, cv_splits=config.cv_splits, cv_test_days=config.cv_test_days
)
pd.DataFrame(cv_windows), holdout_start

## 6. Model Comparison

In [ ]:
results = run_pipeline(config=config, create_notebook=False)
model_comparison = pd.read_csv(config.output_dir / 'model_comparison.csv')
model_comparison

## 7. Error Analysis

In [ ]:
error_analysis = pd.read_csv(config.output_dir / 'error_analysis_by_segment.csv')
error_analysis.head(20)

## 8. Explainability

In [ ]:
display(Image(filename=str(config.output_dir / 'feature_importance_best_model.png')))
shap_path = config.output_dir / 'shap_summary_best_model.png'
if shap_path.exists():
    display(Image(filename=str(shap_path)))
else:
    print("SHAP summary was not generated.")

## 9. Final Training and Submission

In [ ]:
submission = pd.read_csv(config.submission_path)
submission.head(), submission.shape

## 10. Conclusions

- Best model family: `catboost`
- Final holdout window started on `2015-06-06`
- Final model excludes `Customers` to stay forecast-time safe.
- Full comparison and summary live in `outputs/model_comparison.csv` and `outputs/project_summary.md`.